# Problem Statement

- **Role:** You are the Learning & Development (L&D) Head.
- **Goal:** Create a system to generate a curated newsletter for employees.
- **Focus:** Latest advancements in AI Agents and Generative AI.
- **Resource:** Use Analytics Vidhya’s blogs for professional insights and updates.
- **System:** Develop an automated agentic system to:
    - Query Analytics Vidhya for recent articles and news.
    - Curate relevant content on AI Agents and Generative AI.
    - Automatically compile the curated content into a newsletter email.
- **Outcome:** Ensure employees stay informed and aligned with industry advancements, promoting continuous learning.

In [1]:
#install tavily
# pip install tavily-python
#!pip install langchain
#!pip install langchain-community

In [4]:
import autogen
from autogen import ConversableAgent, UserProxyAgent, AssistantAgent
from autogen import register_function
from langchain_community.tools import TavilySearchResults
from autogen.interop import Interoperability
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

In [5]:
from dotenv import load_dotenv
load_dotenv('/Users/janvi/Downloads/AutoGen Handouts/.env')

True

# Sequential Chat with Tool Use

In [6]:
config_list = {
    "config_list": [{"model": "gpt-4o-mini", "temperature": 0.1}],
}

In [7]:
av_blog_agent = ConversableAgent(
    name="av_blog_agent",
    system_message="""You are a helpful AI assistant. You can use the given tool to get the relevant results.
                    Return 'TERMINATE' when the task is done.""",
    llm_config=config_list
)

In [8]:
user_proxy = UserProxyAgent(
    name="User",
    llm_config=False,
    is_termination_msg=lambda msg: msg.get("content") is not None and "TERMINATE" in msg["content"],
    human_input_mode="TERMINATE",
    code_execution_config=False
)

In [9]:
news_agent = ConversableAgent(
    name="news_agent",
    system_message="""You are a helpful AI assistant. You can use the given tool to get the relevant results.
                    Return 'TERMINATE' when the task is done.""",
    llm_config=config_list
)

In [10]:
tavily_tool = TavilySearchResults(max_results=5)

/var/folders/gv/z5kt6xzs54x0jx72s3_02hw80000gp/T/ipykernel_70042/2436168999.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5)


In [11]:
interop = Interoperability()
ag2_tool = interop.convert_tool(tool=tavily_tool, type="langchain")

In [12]:
ag2_tool.register_for_llm(news_agent)
ag2_tool.register_for_execution(user_proxy)

In [13]:
@av_blog_agent.register_for_llm(description="Searches Analytics Vidhya blog to get the results a for given query")
@user_proxy.register_for_execution()
def analytics_vidhya_search():
    """
    Extracts blog headings and descriptions from the given Analytics Vidhya blog archive URL.

    Returns:
        list: A list of dictionaries containing headings and descriptions.
    """

    response = requests.get("https://www.analyticsvidhya.com/blog-archive/")
    soup = BeautifulSoup(response.text, 'html.parser')

    list_items = soup.find_all('li', class_='row list-items')
    results = []

    for item in list_items:
        # Extract heading
        heading = item.find('h3', class_='fs-24 text-light-primary')
        heading_text = heading.text.strip() if heading else "No Heading"

        # Extract link
        url = heading.find_parent('a')['href'] if heading else "No Link"

        # Extract description
        description = item.find('p', class_='fs-16 text-dark-tertiary')
        description_text = description.text.strip() if description else "No Description"

        results.append({
            'heading': heading_text,
            'URL': url,
            'description': description_text,})

    return results

**Step-by-Step Description:**

1. **Function Definition:**  
   `analytics_vidhya_search()` is defined to encapsulate the logic for retrieving and parsing blog data from the Analytics Vidhya blog archive.

2. **Decorators:**  
   - `@av_blog_agent.register_for_llm(description=...)`: Registers the function with an agent (`av_blog_agent`) that interacts with a language model. The description helps define the function's purpose.  
   - `@user_proxy.register_for_execution()`: Enables the function to be executed by the `user_proxy` agent, allowing it to be invoked as needed.

3. **HTTP Request:**  
   The function sends an HTTP `GET` request to the URL `"https://www.analyticsvidhya.com/blog-archive/"` using `requests.get()` to fetch the HTML content of the blog archive.

4. **Parsing HTML:**  
   The returned HTML is parsed using `BeautifulSoup`, creating a `soup` object that represents the DOM structure of the page.

5. **Locating Blog Entries:**  
   The function uses `soup.find_all('li', class_='row list-items')` to locate all `<li>` elements with the class `row list-items`. Each `<li>` corresponds to a blog entry containing a heading, URL, and description.

6. **Extracting Data:**  
   For each blog entry (`<li>` element):  
   - It retrieves the heading from an `<h3>` element with the class `fs-24 text-light-primary`. If no heading exists, it defaults to `"No Heading"`.  
   - It retrieves the URL from the parent `<a>` element of the heading. If no heading exists, it defaults to `"No Link"`.  
   - It retrieves the description from a `<p>` element with the class `fs-16 text-dark-tertiary`. If no description exists, it defaults to `"No Description"`.

7. **Building the Results:**  
   Each heading, URL, and description is stored in a dictionary and appended to the `results` list. This process creates a structured list of all the blog entries.

8. **Return Value:**  
   The function returns the `results` list, which contains dictionaries with `heading`, `URL`, and `description` for each blog entry. This structured output is ready for further processing or display.


In [14]:
email_agent = ConversableAgent(
    name="email_agent",
    system_message="""You can write effective news letter for the given content.
                    Return 'TERMINATE' when the task is done.""",
    llm_config=config_list
)

In [15]:
messages = ["Find the content on AI agents and include the URL in the content",
           """Find the latest news about the topics mentioned in the given content. Result should inlcude both initial content and news results""",
           "Write a news letter email for the given list of artilces and news."]

In [16]:
chat_results = autogen.initiate_chats(
    [
        {
            "sender": user_proxy,
            "recipient": av_blog_agent,
            "message": messages[0],
            "clear_history": True,
            "summary_method": "last_msg",
        },
        {
            "sender": user_proxy,
            "recipient": news_agent,
            "message": messages[1],
            "summary_method": "last_msg",
        },
        {
            "sender": news_agent,
            "recipient": email_agent,
            "message": messages[2],
            "max_turns":1
        },
    ]
)


********************************************************************************
Starting a new chat....

********************************************************************************
User (to av_blog_agent):

Find the content on AI agents and include the URL in the content

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
av_blog_agent (to User):

***** Suggested tool call (call_tWMWDBpYQu9I8nj255iINNzo): analytics_vidhya_search *****
Arguments: 
{}
****************************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION analytics_vidhya_search...
Call ID: call_tWMWDBpYQu9I8nj255iINNzo
Input arguments: {}

>>>>>>>> EXECUTED FUNCTION analytics_vidhya_search...
Call ID: call_tWMWDBpYQu9I8nj255iINNzo
Input arguments: {}
Output:
[{'heading': 'Architecture and Orchestrat

In [17]:
display(Markdown(chat_results[2].summary))

Subject: Explore the Latest Insights on AI Agents!

Dear [Recipient's Name],

We are excited to share a curated selection of articles and news updates that delve into the fascinating world of AI agents. Whether you're a developer, researcher, or simply an enthusiast, these resources will enhance your understanding of the latest trends and technologies in AI.

### Featured Articles:

1. **Architecture and Orchestration of Memory Systems in AI Agents**  
   Discover the principles of memory management in AI agents and how it impacts performance.  
   [Read more here](https://www.analyticsvidhya.com/blog/2026/04/memory-systems-in-ai-agents/)

2. **Replit Agent Skills Complete Guide: Write Your Own Skills in Replit**  
   Learn how to create and implement skills for Replit agents to enhance their capabilities.  
   [Read more here](https://www.analyticsvidhya.com/blog/2026/04/replit-agent-skills/)

3. **Claude Code Leak: 16 Lessons on Building Production-Ready AI Systems**  
   Gain insights from the Claude Code leak and understand the critical aspects of building effective AI systems.  
   [Read more here](https://www.analyticsvidhya.com/blog/2026/04/claude-code-leak-insights-for-ai-builders/)

4. **Qwen3.5-Omni is here! Scaling up to a Native Omni-modal AGI**  
   Explore the capabilities of Qwen3.5-Omni in processing multiple data types simultaneously.  
   [Read more here](https://www.analyticsvidhya.com/blog/2026/03/qwen3-5-omni-ai-model/)

5. **Fine-Tuning vs RAG vs Prompt Engineering**  
   Understand the differences between these methodologies and their applications in AI development.  
   [Read more here](https://www.analyticsvidhya.com/blog/2026/03/fine-tuning-vs-rag-vs-prompt-engineering/)

### Latest News Highlights:

- **Architecting Agent Memory: Principles, Patterns, and Best Practices**  
  A deep dive into memory management strategies for AI agents. [Watch here](https://www.youtube.com/watch?v=W2HVdB4Jbjs)

- **Memory Systems for AI Agents: What the Research Says**  
  Insights into modern agent memory systems and their design dynamics. [Read more here](https://stevekinney.com/writing/agent-memory-systems)

- **Claude Code Leak: 16 Insights for AI Builders**  
  An analysis of the Claude Code leak, focusing on architecture and safety. [Read more here](https://www.analyticsvidhya.com/blog/2026/04/claude-code-leak-insights-for-ai-builders/)

- **Qwen3.5-Omni: Scaling Up, Toward Native Omni-Modal AGI**  
  An introduction to Qwen3.5-Omni and its multimodal processing capabilities. [Read more here](https://qwen.ai/blog?id=qwen3.5-omni)

- **RAG vs fine-tuning vs. prompt engineering - IBM**  
  A discussion on the applications and resource requirements of these methodologies. [Read more here](https://www.ibm.com/think/topics/rag-vs-fine-tuning-vs-prompt-engineering)

We hope you find these articles and updates informative and inspiring as you navigate the evolving landscape of AI agents. 

Best regards,  
[Your Name]  
[Your Position]  
[Your Organization]  
[Your Contact Information]

